# EV spare-parts import prioritisation: scoring, rankings and sensitivity

Vehicles: BYD Seagull, BYD Yuan Up, VW ID.4. Ten part categories, 30 vehicle-category rows.

* Runs top to bottom from a `data/` folder. No network calls and no API keys.
* The factor values come from `data/scoring_input.csv` (built by `make_scoring_input.py` from the cleaned evidence). The written rules are in `data/scoring_rules.csv` (v4: Ethiopian "not found" = neutral 0.5).
* Two rankings: **vehicle-part** (30 rows) and **part-category** (10 rows; the three vehicles combined, weighted by marketplace **listing counts**).
* **Listing counts are listing counts**, not sales, registrations or demand. "Not found" means *not found on the sources inspected*, not scarcity.

In [1]:
# Cell 1 - setup: works locally (VS Code/Jupyter) and on Google Colab.
import os, sys

DATA_DIR = "data"                      # relative path: the data/ folder next to this notebook
IN_COLAB = "google.colab" in sys.modules
REQUIRED = ["scoring_input.csv", "scoring_rules.csv", "weights_proposed.csv",
            "cleaned_vehicle_part_evidence.csv", "vehicle_market_evidence.csv"]

if IN_COLAB:
    # Option A (default): upload the files from the data/ folder when the file picker appears.
    # Option B: put the data/ folder in Google Drive, set USE_DRIVE = True and adjust DRIVE_DATA_DIR.
    USE_DRIVE = False
    DRIVE_DATA_DIR = "/content/drive/MyDrive/ev_spare_parts/data"
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        DATA_DIR = DRIVE_DATA_DIR
    elif not all(os.path.exists(os.path.join(DATA_DIR, f)) for f in REQUIRED):
        from google.colab import files
        os.makedirs(DATA_DIR, exist_ok=True)
        print("Select all files from the data/ folder:", ", ".join(REQUIRED))
        for name, content in files.upload().items():
            with open(os.path.join(DATA_DIR, os.path.basename(name)), "wb") as f:
                f.write(content)

missing = [f for f in REQUIRED if not os.path.exists(os.path.join(DATA_DIR, f))]
assert not missing, f"Missing in {DATA_DIR}/: {missing}"
print("Data folder OK:", DATA_DIR, "| Colab:", IN_COLAB)

Data folder OK: data | Colab: False


In [2]:
import pandas as pd
import numpy as np
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 220)

si = pd.read_csv(os.path.join(DATA_DIR, "scoring_input.csv"))
rules = pd.read_csv(os.path.join(DATA_DIR, "scoring_rules.csv"))
weights_df = pd.read_csv(os.path.join(DATA_DIR, "weights_proposed.csv"))
clean = pd.read_csv(os.path.join(DATA_DIR, "cleaned_vehicle_part_evidence.csv"))

FACTORS = ["demand_evidence", "compatibility", "sourcing_practicality", "scarcity_evidence", "evidence_confidence"]
FCOLS = ["f_" + f for f in FACTORS]
BASE_W = weights_df.set_index("factor")["weight"].reindex(FACTORS)
TOP_N = 5
print(len(si), "rows |", si["vehicle_id"].nunique(), "vehicles |", si["part_category"].nunique(), "categories")

30 rows | 3 vehicles | 10 categories


## 1. Factor rules and weights

In [3]:
display(rules) if "display" in globals() else print(rules)
weights_df

                  factor                                                                                                                     rule  \
0        demand_evidence  Weighted mean of up to 3 sub-scores: V (weight 0.4) = ln(1+n)/ln(1+n_max), where n = the Adey exact-model category L...   
1      scarcity_evidence  Local supply-gap evidence from Batch 06 (Jiji + Adey, first results page): exact-model listing for this part observe...   
2          compatibility  From compatibility_confidence: high 1.0, medium 0.7, low 0.35, insufficient 0.0. No level means year/trim/VIN-confir...   
3  sourcing_practicality  Can an importer act on the international supplier evidence? Specific part record with a displayed price (row or prod...   
4    evidence_confidence  How much evidence stands behind the row (points, then divided by the 0.95 maximum): part number identified +0.30; nu...   

                                                                                                         

,factor,weight,reason
0,demand_evidence,0.30,Main driver of import value: parts for cars that are visible in the market and parts that wear out or get damaged.
1,compatibility,0.20,Importing the wrong part is a direct loss; fitment certainty decides whether a part can be stocked at all.
2,sourcing_practicality,0.20,"A part is only a candidate if a supplier, part number and price exist to act on."
3,scarcity_evidence,0.15,"Lowered on purpose: the Ethiopian evidence is the weakest data (first page of 2 platforms, no stock data)."
4,evidence_confidence,0.15,Lets well-documented rows outrank rows that look good only because data is thin.


## 2. Sanity checks (stop here if any fails)

In [4]:
assert len(si) == 30 and si["record_id"].is_unique, "need 30 unique vehicle-category rows"
assert set(si["record_id"]) == set(clean["record_id"]), "scoring_input does not match the cleaned dataset"
for c in FCOLS:
    vals = si[c].dropna()
    assert ((vals >= 0) & (vals <= 1)).all(), f"{c} outside 0-1"
assert abs(BASE_W.sum() - 1) < 1e-9, "weights must sum to 1"
print("All factors within 0-1; weights sum to", round(BASE_W.sum(), 6))
print("Missing factor values (left empty, never 0):", int(si[FCOLS].isna().sum().sum()))
print("Scarcity values used:", si["f_scarcity_evidence"].value_counts().sort_index().to_dict())

All factors within 0-1; weights sum to 1.0
Missing factor values (left empty, never 0): 0
Scarcity values used: {0.25: 14, 0.375: 4, 0.5: 12}


## 3. Scoring functions

* Vehicle-part score: `sum(weight x factor) / sum(weights of the factors present)`. Missing factors are re-weighted, never set to 0.
* Category score: the three vehicles' scores for that category, combined with **listing-count weights** (Yuan Up 1,362 / Seagull 764 / ID.4 102 Adey listings). These are visibility weights, not sales. The **unweighted mean** is shown as a cross-check.
* Ties are broken by id, so the order is deterministic.

In [5]:
LISTINGS = si.groupby("vehicle_id")["demand_vehicle_listing_count"].first()
assert LISTINGS.to_dict() == {"id4": 102, "seagull": 764, "yuan_up": 1362}, LISTINGS.to_dict()
LW = LISTINGS / LISTINGS.sum()
print("Listing-count weights (share of 2,228 listings):", LW.round(3).to_dict())

def score(df, w):
    w = pd.Series(w, dtype=float).reindex(FACTORS).fillna(0.0)
    vals = df[FCOLS].to_numpy(dtype=float)
    wts = np.tile(w.to_numpy(), (len(df), 1))
    present = ~np.isnan(vals)
    num = np.nansum(vals * wts, axis=1)
    den = (wts * present).sum(axis=1)
    return pd.Series(np.where(den > 0, num / den, np.nan), index=df.index)

def rank_parts(df, w):
    out = df.assign(score=score(df, w)).sort_values(["score", "record_id"], ascending=[False, True]).reset_index(drop=True)
    out["rank"] = np.arange(1, len(out) + 1)
    return out

def rank_categories(df, w):
    s = df.assign(score=score(df, w), lw=df["vehicle_id"].map(LW))
    g = s.groupby("part_category")
    out = pd.DataFrame({
        "score_listing_weighted": g.apply(lambda x: (x["score"] * x["lw"]).sum() / x["lw"].sum(), include_groups=False),
        "score_unweighted_mean": g["score"].mean(),
    }).reset_index()
    out = out.sort_values(["score_listing_weighted", "part_category"], ascending=[False, True]).reset_index(drop=True)
    out["rank"] = np.arange(1, len(out) + 1)
    out["rank_unweighted"] = out["score_unweighted_mean"].rank(ascending=False, method="first").astype(int)
    return out

Listing-count weights (share of 2,228 listings): {'id4': 0.046, 'seagull': 0.343, 'yuan_up': 0.611}


## 4. Rankings with the proposed weights (side by side)

In [6]:
parts = rank_parts(si, BASE_W)
cats = rank_categories(si, BASE_W)
left = parts.head(10)[["rank", "record_id", "score"]].rename(columns={"rank": "part_rank", "score": "part_score"})
right = cats[["rank", "part_category", "score_listing_weighted", "score_unweighted_mean", "rank_unweighted"]].rename(columns={"rank": "cat_rank"})
side = pd.concat([left.reset_index(drop=True), right.reset_index(drop=True)], axis=1)
side.round(3)

,part_rank,record_id,part_score,cat_rank,part_category,score_listing_weighted,score_unweighted_mean,rank_unweighted
0,1,seagull__brake_pad,0.784,1,brake_pad,0.724,0.657,1
1,2,seagull__wiper_blade,0.724,2,fender_body_panel,0.665,0.591,3
2,3,seagull__side_mirror,0.724,3,headlight,0.660,0.555,6
3,4,seagull__tail_light,0.719,4,tail_light,0.659,0.572,4
4,5,seagull__bumper,0.718,5,bumper,0.658,0.543,7
5,6,seagull__fender_body_panel,0.718,6,brake_disc,0.635,0.514,8
6,7,seagull__headlight,0.718,7,shock_absorber,0.633,0.570,5
7,8,id4__wiper_blade,0.716,8,wiper_blade,0.510,0.605,2
8,9,yuan_up__brake_pad,0.709,9,side_mirror,0.449,0.461,9
9,10,seagull__brake_disc,0.700,10,cabin_air_filter,0.367,0.339,10


In [7]:
cols = ["rank", "record_id", "score", "part_numbers", "supplier_price", "ethiopian_availability",
        "et_displayed_prices_etb", "weak_evidence_flags"]
parts.head(TOP_N)[cols].round(3)

,rank,record_id,score,part_numbers,supplier_price,ethiopian_availability,et_displayed_prices_etb,weak_evidence_flags
0,1,seagull__brake_pad,0.784,14758996-00; EQEA-3501500CH,64.0 CNY,listing_observed_exact_model,"Jiji: ETB 28,000",no failure data
1,2,seagull__wiper_blade,0.724,13567800-00; EQEA-5205110,NaN,not_found_on_sources_inspected,NaN,no failure data; no supplier price
2,3,seagull__side_mirror,0.724,13568569-00; EQEA-8202100,429.0 CNY,listing_observed_exact_model,"Jiji: ETB 48,000; Jiji: ETB 55,000",no failure data
3,4,seagull__tail_light,0.719,13541787-00; EQEA-4133010,198.0 CNY,listing_observed_brand_only,NaN,no failure data
4,5,seagull__bumper,0.718,17633728-00; 17633726-00; 18351257-00,NaN,listing_observed_exact_model,NaN,no failure data; single part source; no supplier price


In [8]:
# Evidence behind each top-5 vehicle-part row
for _, r in parts.head(TOP_N).iterrows():
    print(f"#{r['rank']} {r['record_id']}  score={r['score']:.3f}")
    for b in ["demand_basis", "compatibility_basis", "sourcing_basis", "scarcity_basis", "evidence_confidence_basis"]:
        print(f"   {b:<26} {r[b]}")
    print()

#1 seagull__brake_pad  score=0.784
   demand_basis               V=0.92 (764 Adey listings of the vehicle - listing count); C=1.0 (routine_consumable); F missing (no NHTSA data - not sold in US), weights re-normalised
   compatibility_basis        medium (catalogue_model_context_only)
   sourcing_basis             record with displayed price (64.0 CNY); supplier Landin Parts (China)
   scarcity_basis             listing_observed_exact_model; Jiji states 2 ads for the query; 2 exact-model part listings, 1 brand-only (first page, Adey; Jiji)
   evidence_confidence_basis  part no. +0.30, 2nd catalogue +0.10, price +0.10, fitment medium +0.10, ET exact listing +0.15

#2 seagull__wiper_blade  score=0.724
   demand_basis               V=0.92 (764 Adey listings of the vehicle - listing count); C=1.0 (routine_consumable); F missing (no NHTSA data - not sold in US), weights re-normalised
   compatibility_basis        medium (catalogue_model_context_only)
   sourcing_basis             record fou

## 5. Sensitivity analysis (both rankings)

1. Six alternative weight sets.
2. Drop-one-factor: each factor removed in turn, with the remaining weights re-normalised.
3. Random perturbation: every weight multiplied by U(0.8, 1.2), then re-normalised, 2,000 draws, fixed seed.

Rank correlation is Spearman (Pearson on ranks), so no scipy is needed.

In [9]:
ALT = {
    "base (proposed)":   BASE_W.to_dict(),
    "equal weights":     dict.fromkeys(FACTORS, 0.2),
    "demand-heavy":      dict(zip(FACTORS, [0.45, 0.15, 0.15, 0.10, 0.15])),
    "feasibility-heavy": dict(zip(FACTORS, [0.20, 0.30, 0.30, 0.10, 0.10])),
    "scarcity-heavy":    dict(zip(FACTORS, [0.25, 0.15, 0.15, 0.30, 0.15])),
    "evidence-cautious": dict(zip(FACTORS, [0.25, 0.15, 0.15, 0.15, 0.30])),
}
for f in FACTORS:
    w = BASE_W.copy(); w[f] = 0.0
    ALT[f"drop {f}"] = (w / w.sum()).to_dict()

base_p = parts.set_index("record_id")["rank"]; base_c = cats.set_index("part_category")["rank"]
top_p, top_c = list(parts.head(TOP_N)["record_id"]), list(cats.head(TOP_N)["part_category"])
rows, tops_p, tops_c = [], {}, {}
for name, w in ALT.items():
    rp, rc = rank_parts(si, w), rank_categories(si, w)
    tops_p[name] = list(rp.head(TOP_N)["record_id"]); tops_c[name] = list(rc.head(TOP_N)["part_category"])
    rows.append({"scenario": name,
                 "parts_top5_overlap": len(set(tops_p[name]) & set(top_p)),
                 "parts_spearman": round(rp.set_index("record_id")["rank"].reindex(base_p.index).corr(base_p), 3),
                 "cats_top5_overlap": len(set(tops_c[name]) & set(top_c)),
                 "cats_spearman": round(rc.set_index("part_category")["rank"].reindex(base_c.index).corr(base_c), 3),
                 "parts_top5": ", ".join(tops_p[name]), "cats_top5": ", ".join(tops_c[name])})
sens = pd.DataFrame(rows)
sens

,scenario,parts_top5_overlap,parts_spearman,cats_top5_overlap,cats_spearman,parts_top5,cats_top5
0,base (proposed),5,1.000,5,1.000,"seagull__brake_pad, seagull__wiper_blade, seagull__side_mirror, seagull__tail_light, seagull__bumper","brake_pad, fender_body_panel, headlight, tail_light, bumper"
1,equal weights,4,0.975,5,0.988,"seagull__brake_pad, seagull__side_mirror, seagull__tail_light, id4__wiper_blade, seagull__bumper","brake_pad, fender_body_panel, tail_light, headlight, bumper"
2,demand-heavy,4,0.973,5,0.976,"seagull__brake_pad, seagull__wiper_blade, yuan_up__brake_pad, seagull__side_mirror, seagull__bumper","brake_pad, fender_body_panel, headlight, bumper, tail_light"
3,feasibility-heavy,3,0.968,5,0.976,"seagull__brake_pad, seagull__bumper, seagull__fender_body_panel, seagull__headlight, seagull__side_mirror","brake_pad, fender_body_panel, tail_light, headlight, bumper"
4,scarcity-heavy,3,0.965,4,0.842,"seagull__brake_pad, seagull__wiper_blade, id4__wiper_blade, seagull__brake_disc, seagull__tail_light","brake_pad, brake_disc, tail_light, fender_body_panel, headlight"
5,evidence-cautious,3,0.981,5,0.952,"seagull__brake_pad, seagull__side_mirror, seagull__bumper, seagull__fender_body_panel, seagull__headlight","brake_pad, fender_body_panel, bumper, headlight, tail_light"
6,drop demand_evidence,3,0.897,3,0.697,"seagull__brake_disc, seagull__brake_pad, seagull__side_mirror, id4__wiper_blade, seagull__tail_light","brake_disc, brake_pad, fender_body_panel, shock_absorber, tail_light"
7,drop compatibility,4,0.970,5,0.964,"seagull__brake_pad, seagull__wiper_blade, seagull__side_mirror, seagull__tail_light, id4__wiper_blade","brake_pad, tail_light, fender_body_panel, headlight, bumper"
8,drop sourcing_practicality,3,0.969,5,0.976,"seagull__brake_pad, seagull__wiper_blade, seagull__bumper, seagull__fender_body_panel, seagull__headlight","brake_pad, fender_body_panel, headlight, bumper, tail_light"
9,drop scarcity_evidence,3,0.973,5,0.952,"seagull__brake_pad, seagull__side_mirror, seagull__bumper, seagull__fender_body_panel, seagull__headlight","brake_pad, fender_body_panel, bumper, headlight, tail_light"


In [10]:
rng = np.random.default_rng(20261116)
N = 2000
cnt_p = pd.Series(0, index=si["record_id"]); cnt_c = pd.Series(0, index=sorted(si["part_category"].unique()))
for _ in range(N):
    w = BASE_W * rng.uniform(0.8, 1.2, len(FACTORS)); w = w / w.sum()
    cnt_p[rank_parts(si, w).head(TOP_N)["record_id"]] += 1
    cnt_c[rank_categories(si, w).head(TOP_N)["part_category"]] += 1
share_p, share_c = (cnt_p / N).rename("perturb_share_top5"), (cnt_c / N).rename("perturb_share_top5")
pd.concat([share_p[share_p > 0].sort_values(ascending=False).round(3).reset_index().rename(columns={"index": "record_id"}),
           share_c[share_c > 0].sort_values(ascending=False).round(3).reset_index().rename(columns={"index": "part_category"})], axis=1)

,record_id,perturb_share_top5,part_category,perturb_share_top5
0,seagull__brake_pad,1.000,brake_pad,1.0
1,seagull__side_mirror,0.771,bumper,1.0
2,seagull__wiper_blade,0.740,fender_body_panel,1.0
3,seagull__bumper,0.620,headlight,1.0
4,seagull__tail_light,0.523,tail_light,1.0
5,seagull__fender_body_panel,0.500,NaN,NaN
6,id4__wiper_blade,0.351,NaN,NaN
7,seagull__headlight,0.344,NaN,NaN
8,yuan_up__brake_pad,0.149,NaN,NaN
9,seagull__brake_disc,0.002,NaN,NaN


## 6. Robust, borderline and weak-evidence tiers

* **Robust**: in the top 5 in *every* named scenario (6 weight sets + 5 drop-one) **and** in at least 95% of perturbation draws.
* **Borderline**: in the base top 5 but not robust.
* **Weak evidence**: a flag that applies on top of either tier. For a vehicle-part row it means no specific part record, fitment low/insufficient, no supplier price, or Ethiopian factor only "not found". For a category it means the weak flag is set on the vehicle with the largest listing weight in that category.

In [11]:
def tiers(base_top, tops, share, weak):
    out = []
    for k in base_top:
        every = all(k in t for t in tops.values())
        out.append({"item": k, "named_scenarios_in_top5": f"{sum(k in t for t in tops.values())}/{len(tops)}",
                    "perturb_share": round(float(share.get(k, 0)), 3),
                    "tier": "robust" if every and share.get(k, 0) >= 0.95 else "borderline",
                    "weak_evidence": weak.get(k) or "-"})
    return pd.DataFrame(out)

def weak_reason(r):
    rs = []
    flags = str(r["weak_evidence_flags"]) if pd.notna(r["weak_evidence_flags"]) else ""
    for k in ("no specific part record", "fitment low", "fitment insufficient", "no supplier price"):
        if k in flags: rs.append(k)
    if r["ethiopian_availability"] == "not_found_on_sources_inspected": rs.append("ET factor only 'not found'")
    return "; ".join(rs)

weak_p = {r["record_id"]: weak_reason(r) for _, r in si.iterrows()}
lead = si.assign(lw=si["vehicle_id"].map(LW)).sort_values("lw", ascending=False).groupby("part_category").first()
weak_c = {c: (f"{lead.loc[c, 'record_id']}: " + weak_reason(lead.loc[c])) if weak_reason(lead.loc[c]) else "" for c in lead.index}

tier_p = tiers(top_p, tops_p, share_p, weak_p)
tier_c = tiers(top_c, tops_c, share_c, weak_c)
print("VEHICLE-PART top 5"); display(tier_p) if "display" in globals() else print(tier_p)
print("CATEGORY top 5"); tier_c

VEHICLE-PART top 5
                   item named_scenarios_in_top5  perturb_share        tier                                  weak_evidence
0    seagull__brake_pad                   11/11          1.000      robust                                              -
1  seagull__wiper_blade                    6/11          0.740  borderline  no supplier price; ET factor only 'not found'
2  seagull__side_mirror                    8/11          0.771  borderline                                              -
3   seagull__tail_light                    6/11          0.523  borderline                                              -
4       seagull__bumper                    7/11          0.620  borderline                              no supplier price
CATEGORY top 5


,item,named_scenarios_in_top5,perturb_share,tier,weak_evidence
0,brake_pad,11/11,1.0,robust,yuan_up__brake_pad: no supplier price
1,fender_body_panel,11/11,1.0,robust,yuan_up__fender_body_panel: no supplier price
2,headlight,10/11,1.0,borderline,yuan_up__headlight: no supplier price
3,tail_light,11/11,1.0,robust,yuan_up__tail_light: no supplier price
4,bumper,8/11,1.0,borderline,yuan_up__bumper: no supplier price


In [12]:
gap_p = parts["score"].iloc[TOP_N - 1] - parts["score"].iloc[TOP_N]
gap_c = cats["score_listing_weighted"].iloc[TOP_N - 1] - cats["score_listing_weighted"].iloc[TOP_N]
print(f"Vehicle-part: rank 5 vs 6 gap {gap_p:.3f}; ranks 1-10 span {parts['score'].iloc[0]:.3f}-{parts['score'].iloc[9]:.3f}")
print(f"Category:     rank 5 vs 6 gap {gap_c:.3f}")
s5 = parts["score"].iloc[TOP_N - 1]
tied = parts[np.isclose(parts["score"], s5, atol=1e-9)]["record_id"].tolist()
if len(tied) > 1:
    print(f"TIE at the rank-{TOP_N} cut-off (score {s5:.6f}): {tied} - the order between them is only the alphabetical tie-break; treat all as borderline.")
print("Mean top-5 overlap (named scenarios): parts", round(sens["parts_top5_overlap"].mean(), 2), "| categories", round(sens["cats_top5_overlap"].mean(), 2))
no_et_p, no_et_c = set(tops_p["drop scarcity_evidence"]), set(tops_c["drop scarcity_evidence"])
print("Base top-5 parts that leave the top 5 without the Ethiopian factor:", [x for x in top_p if x not in no_et_p] or "none")
print("Base top-5 categories that leave the top 5 without the Ethiopian factor:", [x for x in top_c if x not in no_et_c] or "none")
same_cat_order = (cats["rank"] == cats["rank_unweighted"]).all()
print("Category ranking identical with unweighted mean:", bool(same_cat_order),
      "| top-5 overlap weighted vs unweighted:", len(set(top_c) & set(cats.sort_values("rank_unweighted").head(TOP_N)["part_category"])), "of 5")

Vehicle-part: rank 5 vs 6 gap 0.000; ranks 1-10 span 0.784-0.700
Category:     rank 5 vs 6 gap 0.023
TIE at the rank-5 cut-off (score 0.718130): ['seagull__bumper', 'seagull__fender_body_panel', 'seagull__headlight'] - the order between them is only the alphabetical tie-break; treat all as borderline.
Mean top-5 overlap (named scenarios): parts 3.45 | categories 4.64
Base top-5 parts that leave the top 5 without the Ethiopian factor: ['seagull__wiper_blade', 'seagull__tail_light']
Base top-5 categories that leave the top 5 without the Ethiopian factor: none
Category ranking identical with unweighted mean: False | top-5 overlap weighted vs unweighted: 3 of 5


## 7. How to read this

* The rankings prioritise **evidence**; they are not a sales forecast. Demand uses vehicle *listing counts* as a visibility proxy plus a stated part-replacement assumption.
* "Robust" means the item stays in the top 5 whatever reasonable weights are used. "Borderline" items depend on the weights. "Weak evidence" items need checking before money is spent.
* Before any purchase: confirm fitment by VIN with the supplier, confirm current price and stock, and check Ethiopian prices directly with sellers.